In [ ]:
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
from datasets import load_metric
from transformers import MarianMTModel, MarianTokenizer
from comet import download_model, load_from_checkpoint
import torch


torch.backends.mps.is_available = lambda: False 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)
model.eval()

print(f"Loaded translation model: {model_name}")

df = pd.read_csv("OpenSubtitles_en-fr_clean.csv")
df = df.head(300).reset_index(drop=True)
print(f"Loaded {len(df):,} sentence pairs.")


def translate_chunked_sentence(model, tokenizer, sentence, n, device):
    """
    Split sentence into n-word chunks, translate each, and rejoin.
    """
    words = sentence.split()
    chunks = [" ".join(words[i:i+n]) for i in range(0, len(words), n)]

    translated_chunks = []
    for chunk in chunks:
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True).to(device)
        outputs = model.generate(**inputs, max_length=256)
        translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        translated_chunks.append(translated_text)

    return " ".join(translated_chunks)

context_sizes = [1]
results = {}

for n in context_sizes:
    preds, latencies = [], []
    print(f"\nTranslating with context={n} ...")

    for src in tqdm(df["src"], desc=f"context={n}"):
        start = time.time()
        pred = translate_chunked_sentence(model, tokenizer, src, n, device=device)
        end = time.time()

        preds.append(pred)
        latencies.append(end - start)

    df_context = pd.DataFrame({
        "src": df["src"],
        "tgt": df["tgt"],
        "pred": preds,
        "latency_sec": latencies
    })

    output_path = f"OpenSubtitles_context{n}_translated.csv"
    df_context.to_csv(output_path, index=False, encoding="utf-8")
    print(f"Saved translated results with latency to {output_path}")

    results[n] = df_context


metric_bleu = load_metric("sacrebleu")
metric_chrf = load_metric("chrf")

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

final_scores = []

for n, df_context in results.items():
    print(f"\nEvaluating context={n} ...")

    src_texts = df_context["src"].tolist()
    tgt_texts = df_context["tgt"].tolist()
    preds = df_context["pred"].tolist()

    bleu = metric_bleu.compute(predictions=preds, references=[[t] for t in tgt_texts])
    chrf = metric_chrf.compute(predictions=preds, references=[[t] for t in tgt_texts])

    data = [{"src": s, "mt": p, "ref": t} for s, p, t in zip(src_texts, preds, tgt_texts)]
    comet_score = comet_model.predict(data, batch_size=4, gpus=0, num_workers=0)
    comet_mean = np.mean(comet_score["system_score"])

    avg_latency = df_context["latency_sec"].mean()

    final_scores.append({
        "context": n,
        "BLEU": bleu["score"],
        "chrF": chrf["score"],
        "COMET": comet_mean,
        "Latency (sec/sentence)": avg_latency
    })

    print(
        f"Context={n}: BLEU={bleu['score']:.2f}, chrF={chrf['score']:.2f}, "
        f"COMET={comet_mean:.4f}, Latency={avg_latency:.3f}s"
    )


results_df = pd.DataFrame(final_scores)
results_df.to_csv("OPENSUBTITLES_context=1_evaluation_summary_with_latency.csv", index=False)
print("\n===== Final Evaluation Summary =====")
print(results_df)


/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


✅ Loaded translation model: Helsinki-NLP/opus-mt-en-fr
Loaded 300 sentence pairs.

Translating with context=1 ...


context=1: 100%|██████████| 300/300 [04:34<00:00,  1.09it/s]
/var/folders/00/hlc640bs3hx3jf_rqtdxfpkc0000gn/T/ipykernel_22635/2627510114.py:86: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric_bleu = load_metric("sacrebleu")


✅ Saved translated results with latency to OpenSubtitles_context1_translated_with_latency.csv


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 134432.82it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.5.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
Encoder model frozen.
/Users/choseoyeon/Library/Python/3.9/lib/python/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs



Evaluating context=1 ...


Predicting DataLoader 0: 100%|██████████| 75/75 [00:35<00:00,  2.09it/s]

Context=1: BLEU=2.05, chrF=31.32, COMET=0.4332, Latency=0.914s

===== Final Evaluation Summary =====
   context      BLEU       chrF    COMET  Latency (sec/sentence)
0        1  2.052113  31.321716  0.43318                0.914257
